# Does the Frequent Network keep its 10-minute promise?

> **Status: review in progress.** This notebook was produced by AI under careful guidance, but is currently under human code review.

**The promise.** Between March and December 2025 the CTA phased 20 bus routes into a
"Frequent Network" advertising a bus **every 10 minutes or better**, 6am–9pm weekdays and
9am–9pm weekends. This notebook measures the gaps between buses at each stop, from the times
buses actually passed.

**Why the mean is not the answer.** Riders don't arrive when buses do. Someone walking to a
stop lands in a gap with probability proportional to *that gap's length*, so long gaps catch
more riders than a per-gap average implies. Three buses arriving together on a 10-minute route
leave a 30-minute hole behind them, and the mean headway is still 10. The statistics below are
therefore reported over *riders* as well as over *gaps*. Derivations:
[`docs/methods.md`](docs/methods.md).

The rider-facing object is a survival curve — the share of riders who wait longer than `w`
minutes, equivalently the share of the time the next bus is more than `w` minutes away:

$$S(w) = \frac{\sum_i \max(h_i - w,\ 0)}{\sum_i h_i}$$

---

### How this notebook is organised

- **§2 builds the headways in six visible steps.** Every filter prints what it removed and
  plots what it changed. Nothing is dropped silently.
- **§3 checks the instrument** against raw pings before any statistic is computed.
- **§4–§6 report statistics.** Each cell says what it computes; none says what it means.
- **§7 lists what is unverified, untested, or known to be wrong.**

### Scope

**Route 66 (Chicago Ave) only** — one route end to end, so each step can be checked by hand
before the machinery is pointed at the other 19. Route 66 joined the Frequent Network on
2025-06-15; §4 onward is restricted to that period, since that is when the claim was being
made. This notebook makes **no before/after comparison** — that is a difference statistic and
the project rule is no difference statistic without a null on control routes (§7).

## 0. Setup

In [ ]:
import glob
import os
import itertools
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Shared with data_inventory.ipynb: the palette, style(), the paths into data/,
# and the route/stop loaders. Kept in ctabus.py so the two notebooks cannot end
# up with different versions of the same helper.
import ctabus as cta
from ctabus import (SURFACE, INK, INK2, MUTED, GRID, AXIS,
                    BLUE, ORANGE, AQUA, BLUE_L, style)

cta.apply_style()


In [ ]:
# --- analysis constants, defined once ---
ROUTE = '66'                              # Chicago Ave
PROMISED_HEADWAY_MIN = 10                 # the advertised maximum wait
WEEKDAY_WINDOW = (6, 21)                  # 6am-9pm, the hours the promise covers
WEEKEND_WINDOW = (9, 21)                  # 9am-9pm
PROMISE_BEGINS = pd.Timestamp('2025-06-15')   # route 66 joined the Frequent Network

# Direction is decided by the ORDER in which two patterns traverse their shared
# stops, not by how many they share. Pairs need at least this many shared stops for
# the rank correlation to mean anything.
MIN_SHARED_FOR_DIRECTION = 3
SAME_DIRECTION_CORR = 0.5     # Spearman above this = same direction, below -this = opposite

# Paths come from ctabus.py so they match the other notebook.
ACTUALS_DIR = cta.ACTUALS_DIR
RAW_DIR = cta.RAW_DIR
CACHE_PATH = f'{cta.DERIVED}/headways_rt{ROUTE}.parquet'


## 1. What the data is

**Source.** The Mansueto Institute's [StopWatch](https://github.com/mansueto-institute/cta-stop-watch)
archive, continuing the Chi Hack Night *Ghost Buses* scrape: the CTA Bus Tracker polled every
5 minutes since 2022-05-19, then interpolated onto each route's fixed path to estimate **when
each bus passed each stop**.

One row = one bus visiting one stop:

| column | meaning |
|---|---|
| `bus_stop_time` | estimated time this bus passed this stop |
| `stpid` | the stop |
| `stop_sequence` | position along the pattern |
| `pid` | pattern (one specific path along the route) |
| `speed_mph` | how fast the bus was moving |

`bus_stop_time` is **interpolated, not observed** — the scrape records vehicle positions, not
stop crossings. §3 tests the instrument against raw pings rather than assuming it.

> Note that StopWatch's own published analysis covers June 2022 – July 2024. Everything
> after that, is outside the period they validated, but appears to have continued to run well.

In [ ]:
# Find this route's pattern files. Every actuals file carries a constant `rt`
# column, so the route -> pattern map is read straight off the local files.
pattern_files = []
for path in sorted(glob.glob(f'{ACTUALS_DIR}/*.parquet')):
    first_row = pq.ParquetFile(path).read_row_group(0, columns=['rt']).slice(0, 1).to_pylist()
    if first_row[0]['rt'] == ROUTE:
        pattern_files.append(path)

print(f'route {ROUTE}: {len(pattern_files)} pattern files')
for path in pattern_files:
    print(f'  {os.path.basename(path):<32} {pq.ParquetFile(path).metadata.num_rows:>10,} rows')

### Only route 66 buses are counted

Each actuals file covers exactly one route, so nothing from another route can enter these
headways. But many of route 66's stops are also served by other routes, and the cell below
counts them.

**Those other buses are deliberately not counted.** A 65 stopping at a shared corner is no use
to someone travelling along Chicago Ave past the point where the 65 diverges — the routes
share a stop, not a destination. The Frequent Network promise is also made per route. So the
quantity measured here is *the wait for a route 66 bus*, and the count below records how often
a rider at these stops would have seen some other bus go by in the meantime.

In [ ]:
# Which other routes touch route 66's stops? Scans every pattern file, so it is the
# slowest cell in the notebook -- cached after the first run.
SHARED_STOPS_CACHE = 'data/derived/rt66_shared_stops.csv'

if os.path.exists(SHARED_STOPS_CACHE):
    shared = pd.read_csv(SHARED_STOPS_CACHE, dtype={'route': str})
else:
    route_66_stops = set()
    for path in pattern_files:
        route_66_stops |= set(pq.read_table(path, columns=['stpid']).to_pandas().stpid.unique())
    rows = []
    for path in sorted(glob.glob(f'{ACTUALS_DIR}/*.parquet')):
        head = pq.ParquetFile(path).read_row_group(0, columns=['rt']).slice(0, 1).to_pylist()[0]
        if head['rt'] == ROUTE:
            continue
        other_stops = set(pq.read_table(path, columns=['stpid']).to_pandas().stpid.unique())
        common = route_66_stops & other_stops
        if common:
            rows.append({'route': head['rt'], 'shared_stops': len(common),
                         'stops': '|'.join(sorted(common))})
    shared = (pd.DataFrame(rows).groupby('route')
              .agg(shared_stops=('stops', lambda s: len(set('|'.join(s).split('|')))))
              .reset_index().sort_values('shared_stops', ascending=False))
    os.makedirs('data/derived', exist_ok=True)
    shared.to_csv(SHARED_STOPS_CACHE, index=False)

print(f'other routes touching at least one route-66 stop: {len(shared)}')
print(shared.head(12).to_string(index=False))

### What route 66 looks like

Before any filtering, a look at the route itself: where its stops are, and how its 13 patterns
lay out along Chicago Ave.

The actuals carry no coordinates, only `stpid`. Locations come from GTFS `stops.txt`, joined
on the stop id, and the match rate is printed — the GTFS feed is one snapshot while the
arrivals run from 2022, so stops retired before that snapshot are not found.

The plotting helpers are in [`ctabus.py`](ctabus.py) and take any route;
[`data_inventory.ipynb`](data_inventory.ipynb) §4 uses the same three on a route of your
choosing.

Two things here bear on §2 rather than being decided by it:

- **`stop_sequence` against longitude** separates the patterns into two fans with opposite
  slopes. §2 step 2 works out direction arithmetically from traversal order; this is the
  picture of the same thing, and the two should agree.
- **Several patterns cover only part of the route.** Those are short-turns, and they are why
  step 4's terminal rule is applied per pattern rather than per route.


In [ ]:
route_stops = cta.route_stops(ROUTE)
located = route_stops.stop_lat.notna()

print(f'pattern-stop rows        : {len(route_stops):,}')
print(f'distinct patterns        : {route_stops.pid.nunique()}')
print(f'distinct stops           : {route_stops.stpid.nunique()}')
print(f'rows with GTFS coords    : {located.sum():,} ({located.mean():.1%})')
print(f'distinct stpid unmatched : {route_stops.loc[~located, "stpid"].nunique()}  '
      f'{sorted(route_stops.loc[~located, "stpid"].unique())}')

cta.plot_route_overview(route_stops, ROUTE)
plt.tight_layout()
plt.show()


In [ ]:
cta.plot_pattern_panels(route_stops, ROUTE)
plt.show()

cta.plot_sequence_vs_longitude(route_stops, ROUTE)
plt.tight_layout()
plt.show()


## 2. Building the headways, one visible step at a time

Six operations turn ~31 million stop visits into a list of gaps. Each prints what it removed
and plots what it changed.

### Step 1 — load every pattern on the route, and drop the two partial days

A "pattern" is one path a bus can take: the full route, a short-turn, a variant skipping a
segment. All are loaded, on the reasoning that a rider boards whatever comes and cannot see
which pattern a bus is running.

The coverage trace comes first, then the one filter this step applies. **The first and last
calendar days in the archive are dropped, because neither is a whole day of service.** The
archive begins part-way through its first day and ends wherever the pipeline last ran. Left
in, both would look like thin service and would contribute gaps spanning hours in which
nothing was recorded.

**As the data currently stands this filter removes nothing that survives to §4.** All 430
visits on those two days fall in hours 23, 00 and 01, so the step 5 window (6am–9pm) already
excluded every one of them — the cell prints the hours so this can be checked rather than
assumed. The headline statistics are identical with and without it.

It is kept as a guard rather than a correction. It costs two days out of ~1,500, and it stops
a future archive whose last day ends mid-afternoon from quietly contributing a partial day to
the middle of the promise window.


In [ ]:
frames = [pq.read_table(path, columns=['stpid', 'bus_stop_time', 'stop_sequence',
                                       'pid', 'speed_mph']).to_pandas()
          for path in pattern_files]

stop_visits = pd.concat(frames, ignore_index=True)
stop_visits['stpid'] = stop_visits.stpid.astype('category')
stop_visits['pid'] = stop_visits.pid.astype('category')
stop_visits['speed_mph'] = stop_visits.speed_mph.astype('float32')
del frames

print(f'{len(stop_visits):,} stop visits')
print(f'{stop_visits.bus_stop_time.min():%Y-%m-%d} to {stop_visits.bus_stop_time.max():%Y-%m-%d}')
print(f'{stop_visits.stpid.nunique()} distinct stops, {stop_visits.pid.nunique()} patterns')

In [ ]:
# Coverage. A day with far fewer visits than its neighbours would be a scrape gap
# rather than a service cut, and would appear downstream as a very long headway.
visits_per_day = stop_visits.bus_stop_time.dt.normalize().value_counts().sort_index()
span_days = (visits_per_day.index.max() - visits_per_day.index.min()).days + 1

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(visits_per_day.index, visits_per_day.values, lw=0.7, color=BLUE)
ax.set_title(f'Route {ROUTE}: stop visits recorded per day')
style(ax, 'visits / day')
plt.show()

print(f'days present in data : {len(visits_per_day):,}')
print(f'calendar days in span: {span_days:,}')
print(f'missing days         : {span_days - len(visits_per_day):,}')
print(f'\nthinnest 5 days:')
print(visits_per_day.nsmallest(5).to_string())

**To inspect:** whether the trace has step changes or dropouts, and whether the thinnest
days are genuine partial days (e.g. the first day of the scrape) or something else.

In [ ]:
# Drop the first and last calendar day: both are partial, so their visit counts are
# not comparable with a full day's and any gap they produce spans unrecorded hours.
visit_date = stop_visits.bus_stop_time.dt.normalize()
first_day, last_day = visit_date.min(), visit_date.max()
typical = visits_per_day.median()

print(f'median full day: {typical:,.0f} visits\n')
for label, day in [('first', first_day), ('last', last_day)]:
    n = visits_per_day.loc[day]
    print(f'  {label:>5} day {day:%Y-%m-%d}: {n:>7,} visits   '
          f'{n / typical:>5.0%} of a median day')

is_partial_day = visit_date.isin([first_day, last_day])

# Which hours these visits sit in decides whether this filter removes anything the
# step 5 window would not have removed anyway. Printed rather than assumed.
dropped_hours = stop_visits.loc[is_partial_day, 'bus_stop_time'].dt.hour
inside_window = ((dropped_hours >= WEEKDAY_WINDOW[0]) &
                 (dropped_hours < WEEKDAY_WINDOW[1])).sum()
print(f'\nhours they fall in : {dropped_hours.value_counts().sort_index().to_dict()}')
print(f'inside {WEEKDAY_WINDOW[0]}:00-{WEEKDAY_WINDOW[1]}:00   : {inside_window}'
      f'   <- if 0, step 5 would have removed them all regardless')

print(f'\nvisits removed: {is_partial_day.sum():,} ({is_partial_day.mean():.3%})')
stop_visits = stop_visits[~is_partial_day].copy()
remaining = stop_visits.bus_stop_time.dt.normalize()
print(f'remaining     : {len(stop_visits):,}')
print(f'span is now   : {remaining.min():%Y-%m-%d} to {remaining.max():%Y-%m-%d}  '
      f'({remaining.nunique():,} days)')


### Step 2 — determine direction from the order stops are traversed

The actuals carry **no direction column**, and no reliable external label exists (see the note
at the end of this step). Direction is therefore measured from the data.

**The test.** Take two patterns and the stops they have in common. Rank each pattern's
`stop_sequence` over those shared stops and correlate the ranks. If both patterns pass the same
stops in the same order they run the same direction (Spearman ≈ +1); if one passes them in
reverse, they run opposite directions (≈ −1). This measures direction of travel directly rather
than inferring it from how *many* stops two patterns share.

Patterns are then grouped by same-direction edges, and the grouping is **checked against the
opposite-direction pairs**: any two patterns found to run opposite directions must land in
different groups. That check can fail and will say so.

*Why not simply count shared stops:* an earlier version joined patterns sharing any stop at
all. Three terminal-loop stops are used by both directions, which welded all 13 patterns into
one group — and the check then compared that single group against nothing and printed a pass.
It could not fail. Raising the count to a threshold fixes the symptom but leaves an arbitrary
constant deciding direction.

In [ ]:
# Each pattern's position along its own path, per stop.
sequence_by_pattern = {pattern: group.groupby('stpid', observed=True).stop_sequence.median()
                       for pattern, group in stop_visits.groupby('pid', observed=True)}
stops_by_pattern = {p: set(s.index) for p, s in sequence_by_pattern.items()}
patterns = sorted(stops_by_pattern, key=lambda p: -len(stops_by_pattern[p]))

# Spearman rank correlation of stop order over each pair's shared stops.
direction_corr = pd.DataFrame(np.nan, index=patterns, columns=patterns, dtype=float)
shared_count = pd.DataFrame(0, index=patterns, columns=patterns, dtype=int)
for a, b in itertools.combinations(patterns, 2):
    common = sorted(stops_by_pattern[a] & stops_by_pattern[b])
    shared_count.loc[a, b] = shared_count.loc[b, a] = len(common)
    if len(common) >= MIN_SHARED_FOR_DIRECTION:
        corr = sequence_by_pattern[a][common].corr(
            sequence_by_pattern[b][common], method='spearman')
        direction_corr.loc[a, b] = direction_corr.loc[b, a] = corr

fig, ax = plt.subplots(figsize=(6.8, 5.6))
im = ax.imshow(direction_corr.values, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(len(patterns))); ax.set_xticklabels(patterns, rotation=90, fontsize=7)
ax.set_yticks(range(len(patterns))); ax.set_yticklabels(patterns, fontsize=7)
for i, a in enumerate(patterns):
    for j, b in enumerate(patterns):
        if i != j and shared_count.loc[a, b]:
            value = direction_corr.loc[a, b]
            ax.text(j, i, 'n/a' if pd.isna(value) else f'{value:+.1f}',
                    ha='center', va='center', fontsize=6, color=INK)
ax.set_title(f'Route {ROUTE}: Spearman order correlation over shared stops')
fig.colorbar(im, ax=ax, shrink=0.7, label='+1 same direction   −1 opposite')
ax.grid(False)
plt.show()

pairs_with_shared = (shared_count.values > 0).sum() // 2
print(f'pattern pairs sharing at least one stop      : {pairs_with_shared}')
print(f'pairs with too few shared stops to correlate : '
      f'{int(((shared_count.values > 0) & direction_corr.isna().values).sum() // 2)}')

In [ ]:
# Group patterns by same-direction edges only.
parent = {p: p for p in patterns}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

opposite_pairs = []
for a, b in itertools.combinations(patterns, 2):
    corr = direction_corr.loc[a, b]
    if pd.isna(corr):
        continue
    if corr >= SAME_DIRECTION_CORR:
        parent[find(a)] = find(b)
    elif corr <= -SAME_DIRECTION_CORR:
        opposite_pairs.append((a, b, corr))

components = {}
for pattern in patterns:
    components.setdefault(find(pattern), []).append(pattern)
components = list(components.values())
component_of_pattern = {p: i for i, members in enumerate(components) for p in members}

print(f'direction groups found: {len(components)}')
stops_per_component = []
for i, members in enumerate(components):
    member_stops = set().union(*(stops_by_pattern[p] for p in members))
    stops_per_component.append(member_stops)
    print(f'  group {i}: {len(members):>2} patterns, {len(member_stops):>3} stops  {members}')

# CHECK THAT CAN FAIL: patterns measured as running opposite directions must not
# have been placed in the same group.
violations = [(a, b, c) for a, b, c in opposite_pairs
              if component_of_pattern[a] == component_of_pattern[b]]
print(f'\nopposite-direction pairs detected : {len(opposite_pairs)}')
print(f'of those, wrongly grouped together: {len(violations)}   <- must be 0')
for a, b, c in violations:
    print(f'   VIOLATION {a} vs {b}: corr {c:+.2f}')

In [ ]:
# Stops used by more than one direction group. Sorting arrivals by time at such a
# stop interleaves opposing buses, so the differences are not headways for any rider.
stop_group_count = pd.Series(0, index=sorted(set().union(*stops_per_component)))
for member_stops in stops_per_component:
    stop_group_count[sorted(member_stops)] += 1
both_direction_stops = set(stop_group_count[stop_group_count > 1].index)

print(f'stops served by more than one direction group: {len(both_direction_stops)} of '
      f'{len(stop_group_count)} ({len(both_direction_stops)/len(stop_group_count):.1%})')
print(f'  {sorted(both_direction_stops)}')

**On the external labels that were considered and rejected.** Two explicit direction
sources exist and neither is usable here:

- **The destination sign** (`des` in the raw pings) gives one destination per pattern. It
  disagrees with the measured geometry for pattern `6982`, which is signed *Pulaski*
  (westbound) on all of its pings but traverses its 22 shared stops in the **same** order as
  the eastbound `6662` (Spearman +1.00). The likely cause is operators setting the return
  destination while finishing an eastbound leg. §3 prints this cross-check.
- **GTFS `trips.txt`** carries a real `direction` field, but joining it to the timetables on
  `schd_trip_id` returns *both* directions for nearly every pattern — those ids are not unique
  across a multi-year accumulated timetable. The join is unusable, not merely noisy.

The authoritative source would be the Bus Tracker `getpatterns` endpoint, which returns
`rtdir` per pattern. It needs a live API key, so it is not available offline.

### Step 3 — remove stops served in both directions

At a stop used by both directions, sorting arrivals by time interleaves opposing buses, so the
differences are not headways for any rider. Those stops are removed. The cell reports how much
data that costs and how those stops differed, so the choice can be reversed and re-examined.

An alternative — splitting each such stop by the direction of the bus rather than dropping the
stop — is not implemented here. It is listed in §7.

In [ ]:
before = len(stop_visits)
is_both_direction = stop_visits.stpid.isin(both_direction_stops)
print(f'stops removed        : {len(both_direction_stops)}')
print(f'stop visits removed  : {is_both_direction.sum():,} ({is_both_direction.mean():.2%})')

stop_visits = stop_visits[~is_both_direction].copy()
print(f'remaining            : {len(stop_visits):,}')

### Step 4 — drop terminal stops

A bus on layover at the end of the line still broadcasts its position, so it can enter the
data as an arrival nobody could board. The first and last stop of each pattern are removed.

Note this is a *per-pattern* rule: a stop that is a terminal for a short-turn pattern but
mid-route for the full pattern is dropped only for the rows belonging to the short-turn.

In [ ]:
sequence_bounds = stop_visits.groupby('pid', observed=True).stop_sequence.agg(['min', 'max'])
sequence_dtype = stop_visits.stop_sequence.dtype
at_terminal = (
    (stop_visits.stop_sequence == stop_visits.pid.map(sequence_bounds['min']).astype(sequence_dtype)) |
    (stop_visits.stop_sequence == stop_visits.pid.map(sequence_bounds['max']).astype(sequence_dtype))
)

print(f'terminal visits removed: {at_terminal.sum():,} ({at_terminal.mean():.1%})')
stop_visits = stop_visits[~at_terminal].copy()
print(f'remaining              : {len(stop_visits):,}')

print('\nspeed_mph at the surviving visits:')
print(stop_visits.speed_mph.describe([.01, .05, .5, .95, .99]).round(2).to_string())

**On `speed_mph`, and why no parked-bus rule is applied.** The data plan carried over a rule
from the transit-insights project to drop parked buses by speed. That rule cannot do anything
on this data, and the percentiles above cannot detect whether it should.

StopWatch clips the column before we ever see it —
[`interpolation.py:110-115`](https://github.com/mansueto-institute/cta-stop-watch/blob/main/cta-stop-watch/report_automation/interpolation.py):

```python
# replace values below 1 or above 115 for speed_mph
stops_df["speed_mph"] = stops_df["speed_mph"].apply(
    lambda x: np.nan if x < 1 or x > 115 else x)
stops_df["speed_mph"] = stops_df["speed_mph"].fillna(method="ffill")
stops_df["speed_mph"] = stops_df["speed_mph"].fillna(method="bfill")
```

So the minimum is 1 by construction — the `min` printed above is exactly that floor, not a
measurement — and any value that *was* below 1 has been overwritten with a neighbouring row's
speed. A stationary bus is therefore indistinguishable from a moving one in this column.

Two further points, from the same file: `speed_mph` is the average over a whole ping-to-ping
segment broadcast to every stop in it (`:32`, `:81`), not a speed at the stop; and it is used
to *extrapolate* `bus_stop_time` for stops beyond a trip's first and last ping (`:145-155`),
which is an independent reason to drop terminal stops as step 4 does.

`data_inventory.ipynb` §2c plots the column.


### Step 5 — restrict to the hours the promise covers

6am–9pm weekdays, 9am–9pm weekends. Filtering *before* differencing is what makes every gap
have both ends inside the window: if the overnight hours stayed in, the interval from the last
bus near midnight to the first before 6am would enter as a single ~300-minute headway.

In [ ]:
stop_visits['service_date'] = stop_visits.bus_stop_time.dt.normalize()
stop_visits['hour'] = stop_visits.bus_stop_time.dt.hour
stop_visits['is_weekend'] = stop_visits.bus_stop_time.dt.dayofweek >= 5

window_opens = np.where(stop_visits.is_weekend, WEEKEND_WINDOW[0], WEEKDAY_WINDOW[0])
window_closes = np.where(stop_visits.is_weekend, WEEKEND_WINDOW[1], WEEKDAY_WINDOW[1])
in_promised_window = (stop_visits.hour >= window_opens) & (stop_visits.hour < window_closes)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, is_weekend, label, window in [(axes[0], False, 'Weekday', WEEKDAY_WINDOW),
                                      (axes[1], True, 'Weekend', WEEKEND_WINDOW)]:
    subset = stop_visits[stop_visits.is_weekend == is_weekend]
    by_hour = subset.groupby('hour').size() / subset.service_date.nunique()
    ax.bar(by_hour.index, by_hour.values, color=BLUE_L, width=0.85)
    kept = (by_hour.index >= window[0]) & (by_hour.index < window[1])
    ax.bar(by_hour.index[kept], by_hour.values[kept], color=BLUE, width=0.85)
    ax.set_title(f'{label} — promise covers {window[0]}:00–{window[1]}:00')
    ax.set_xlabel('hour of day')
    style(ax, 'visits per day' if not is_weekend else None)
plt.show()

print(f'visits outside the window removed: {(~in_promised_window).sum():,} '
      f'({(~in_promised_window).mean():.1%})')
stop_visits = stop_visits[in_promised_window].copy()
print(f'remaining                        : {len(stop_visits):,}')

### Step 6 — sort, then difference

Within each **(stop, day)**, sort arrivals by time and take successive differences. The first
bus of each day yields no headway, which `diff()` marks as missing.

Grouping by day assumes no gap crosses midnight — which holds only because the window closes
at 21:00.

In [ ]:
stop_visits = stop_visits.sort_values(['stpid', 'service_date', 'bus_stop_time'])
gap = stop_visits.groupby(['stpid', 'service_date'], observed=True).bus_stop_time.diff()
stop_visits['headway_min'] = gap.dt.total_seconds() / 60

headways = stop_visits.dropna(subset=['headway_min']).copy()
print(f'headways                  : {len(headways):,}')
print(f'first-of-day visits dropped: {len(stop_visits) - len(headways):,}')
print(f'from {headways.stpid.nunique()} stops over {headways.service_date.nunique():,} days')

h_check = headways.headway_min.values
print(f'\nstrictly negative gaps: {(h_check < 0).sum():,}   (sorting should force 0)')
print(f'exactly zero gaps     : {(h_check == 0).sum():,}   (two buses logged at the same instant)')
print(f'median {np.median(h_check):.2f} min | mean {h_check.mean():.2f} min | '
      f'p95 {np.percentile(h_check, 95):.1f} | max {h_check.max():.0f}')

In [ ]:
# Cache so the notebook can be re-run from here without the multi-minute load.
os.makedirs('data/derived', exist_ok=True)
headways[['stpid', 'service_date', 'bus_stop_time', 'hour', 'is_weekend', 'headway_min']] \
    .to_parquet(CACHE_PATH, index=False)
print(f'wrote {CACHE_PATH}')

## 3. Instrument check against raw pings

`bus_stop_time` is interpolated. Before computing anything, one question: **when this pipeline
reports a long gap, is that a bus that never came, or a bus the scrape missed?**

Long gaps in this archive are concentrated in 2022. The cells below compare one week of raw
pings from October 2022 against the matched week of October 2024 — same season, so the
calendar is held roughly constant.

Three quantities are compared: how often each bus was re-observed, how many stop crossings the
interpolation produced per vehicle, and how many vehicles were on the street.

> **Known weakness, stated before the output.** `visits_per_vehicle` divides *in-window,
> filtered headway rows* by *distinct vehicles across the whole day, all hours*. The numerator
> and denominator have different scopes, so the ratio is only a rough proxy and is biased if
> the share of service outside 6a–9p differs between the two eras. `revisit_median_min`,
> `vehicles` and `gaps_over_30min` do not have this problem. Rebuilding the ratio with matched
> scopes is in §7.

> **Sample size:** two weeks, both in October. This is not a coverage study.

In [ ]:
# Raw ping days on disk (fetched with fetch_stopwatch.py --what raw).
raw_days = {}
for path in sorted(glob.glob(f'{RAW_DIR}/*.csv')):
    day = pd.read_csv(path, usecols=['vid', 'rt', 'pid', 'des', 'data_time'],
                      dtype={'rt': str, 'vid': str, 'des': str})
    day['data_time'] = pd.to_datetime(day.data_time)
    raw_days[os.path.basename(path)[:-4]] = day

print(f'{len(raw_days)} raw days loaded: {sorted(raw_days)[0]} … {sorted(raw_days)[-1]}')

In [ ]:
# Per-vehicle re-observation interval is what constrains the interpolation --
# NOT the number of distinct poll timestamps in the file, which counts every route
# and is much larger because routes are polled at different offsets.
instrument = []
for day_label, raw in sorted(raw_days.items()):
    route_pings = raw[raw.rt == ROUTE]
    revisit = (route_pings.sort_values(['vid', 'data_time'])
               .groupby('vid').data_time.diff().dt.total_seconds().div(60).dropna())
    same_day = headways[headways.service_date == pd.Timestamp(day_label)]
    instrument.append({
        'day': day_label,
        'era': day_label[:4],
        'poll_minutes_all_routes': raw.data_time.nunique(),
        'poll_minutes_this_route': route_pings.data_time.nunique(),
        'vehicles': route_pings.vid.nunique(),
        'revisit_median_min': revisit.median(),
        'revisit_p99_min': revisit.quantile(0.99),
        'headway_rows': len(same_day),
        'visits_per_vehicle': len(same_day) / max(route_pings.vid.nunique(), 1),
        'gaps_over_30min': int((same_day.headway_min > 30).sum()),
    })
instrument = pd.DataFrame(instrument)
print(instrument.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.3))
era_colour = {'2022': ORANGE, '2024': BLUE}
positions = {'2022': range(0, 7), '2024': range(8, 15)}

for ax, column, title in [
    (axes[0], 'revisit_median_min', 'Per-vehicle\nre-observation (min)'),
    (axes[1], 'vehicles', 'Distinct vehicles\non the route'),
    (axes[2], 'visits_per_vehicle', 'Headway rows per vehicle\n(scope-mismatched)'),
    (axes[3], 'gaps_over_30min', 'Gaps over 30 min\nper day'),
]:
    for era, group in instrument.groupby('era'):
        ax.bar(list(positions[era]), group[column].values,
               color=era_colour[era], width=0.8, label=era)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([3, 11]); ax.set_xticklabels(['2022', '2024'])
    style(ax)
axes[0].legend()
plt.tight_layout()
plt.show()

print(instrument.groupby('era')[['revisit_median_min', 'revisit_p99_min', 'vehicles',
                                 'visits_per_vehicle', 'gaps_over_30min']].mean().round(1).to_string())

**To inspect:** whether `revisit_median_min` differs between the eras (an instrument
change), whether `vehicles` differs (a service change), and whether `gaps_over_30min` tracks
one or the other. Those three together are what distinguish a scrape artefact from a real
service failure — and the second panel's answer bears directly on whether 2022 should be
excluded from any analysis that uses it.

**Context for interpretation, not a conclusion:** October 2022 falls inside the period of
CTA's operator shortage, when missed runs were a documented public problem. Whether the
numbers above reflect that is the reader's call.

In [ ]:
# Cross-check the measured direction groups (step 2) against the destination sign
# carried in the raw pings. Disagreements are printed, not resolved.
signs = pd.concat([raw[raw.rt == ROUTE][['pid', 'des']] for raw in raw_days.values()])
signs = signs.dropna(subset=['pid'])
signs['pattern'] = signs.pid.astype(float).astype(int).astype(str)
sign_of_pattern = (signs.groupby('pattern').des
                   .agg(lambda s: s.value_counts().idxmax()))

rows = []
for pattern, group_index in component_of_pattern.items():
    key = str(int(float(pattern)))
    rows.append({'pattern': pattern, 'direction_group': group_index,
                 'destination_sign': sign_of_pattern.get(key, '(not in raw sample)')})
cross_check = pd.DataFrame(rows).sort_values(['direction_group', 'pattern'])
print(cross_check.to_string(index=False))

print('\nsigns appearing in MORE THAN ONE measured direction group:')
by_sign = cross_check[cross_check.destination_sign != '(not in raw sample)']
conflicted = by_sign.groupby('destination_sign').direction_group.nunique()
conflicted = conflicted[conflicted > 1]
print(f'  {list(conflicted.index) if len(conflicted) else "none"}')
print('\n(A sign in two groups means the sign and the traversal order disagree for at')
print(' least one pattern. The traversal order is what step 2 uses.)')

In [ ]:
# Do any route-66 patterns in the raw pings have no processed file? Those are
# service the actuals cannot see at all.
have_patterns = {str(int(float(p))) for p in patterns}
for era in ['2022', '2024']:
    era_raw = pd.concat([raw for day, raw in raw_days.items() if day.startswith(era)])
    route_pings = era_raw[era_raw.rt == ROUTE].copy()
    route_pings['pid_str'] = route_pings.pid.dropna().astype(float).astype(int).astype(str)
    counts = route_pings.pid_str.value_counts()
    absent = [p for p in counts.index if p not in have_patterns]
    share = counts[absent].sum() / counts.sum() if absent else 0.0
    print(f'{era}: {len(counts)} patterns in raw pings, {len(absent)} with no processed file '
          f'{absent} -> {share:.1%} of pings')

## 4. The distribution of gaps

From here the data is restricted to **weekdays from 2025-06-15**, the period during which the
10-minute claim applied to route 66.

In [ ]:
promise_period = headways[(~headways.is_weekend) &
                          (headways.service_date >= PROMISE_BEGINS)]
h = promise_period.headway_min.values
h = h[h > 0]          # exact ties carry no wait and would divide by zero below

mean_headway = h.mean()
cv = h.std() / mean_headway
print(f'n              {len(h):,} headways, weekdays from {PROMISE_BEGINS:%Y-%m-%d}')
print(f'mean headway   {mean_headway:.2f} min')
print(f'median headway {np.median(h):.2f} min')
print(f'CV (sd/mean)   {cv:.2f}      (0 = perfectly even spacing)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(h[h < 40], bins=np.arange(0, 40, 0.5), color=BLUE_L)
ax.axvline(mean_headway, color=BLUE, lw=2, label=f'mean {mean_headway:.1f} min')
ax.axvline(PROMISED_HEADWAY_MIN, color=ORANGE, lw=2, ls='--',
           label=f'{PROMISED_HEADWAY_MIN}-minute promise')
ax.set_title(f'Route {ROUTE}: gaps between buses (weekdays, in-window, promise period)')
ax.set_xlabel('minutes between one bus and the next')
ax.legend()
style(ax, 'count')
plt.show()

for threshold in [1, 2, 5, PROMISED_HEADWAY_MIN, 15, 20, 30]:
    print(f'  share of gaps under {threshold:>2} min: {(h < threshold).mean():>6.1%}'
          f'      over {threshold:>2} min: {(h > threshold).mean():>6.1%}')

**To inspect:** the mass near zero relative to the mass beyond 10 minutes, and where the
mean sits relative to the promise. Under the length-biasing described in the header, those two
tails are not independent of one another.

In [ ]:
# Reference point: how much service would be needed at exactly the promised
# headway to cover the same total observed time.
buses_observed = len(h)
buses_at_promised_headway = h.sum() / PROMISED_HEADWAY_MIN

print(f'total time spanned by the observed gaps : {h.sum():,.0f} bus-minutes')
print(f'observed number of gaps                 : {buses_observed:,}')
print(f'gaps needed to span the same time at')
print(f'  exactly {PROMISED_HEADWAY_MIN} minutes each               : {buses_at_promised_headway:,.0f}')
print(f'ratio                                   : '
      f'{buses_at_promised_headway / buses_observed:.2f}')
print(f'\nS(10) for perfectly even service at the observed mean of '
      f'{mean_headway:.2f} min: {max(mean_headway - PROMISED_HEADWAY_MIN, 0) / mean_headway:.3f}')

## 5. The rider's curve

`S(w)` is the share of riders who wait more than `w` minutes — equivalently the share of the
time the next bus is more than `w` minutes away. Both readings come from the same sum: the
numerator $\sum_i \max(h_i - w, 0)$ counts minutes during which the next bus is more than
`w` minutes off, and riders arriving uniformly land in those minutes in proportion to how many
there are. This rests on the uniform-arrivals assumption stated in
[`docs/methods.md`](docs/methods.md), which is weakest late at night and on infrequent
routes.

In [ ]:
def survival(headway_minutes, w):
    """S(w): share of riders waiting longer than w minutes.

    Also the share of in-window time during which the next bus is more than w
    minutes away. Derivation in docs/methods.md.
    """
    return np.maximum(headway_minutes - w, 0).sum() / headway_minutes.sum()


def even_service_survival(mean_headway, w):
    """S(w) if the same total service ran at a perfectly constant headway."""
    return max(mean_headway - w, 0) / mean_headway


# Implementation check from docs/methods.md: the area under S(w) equals the mean
# rider wait computed directly. This tests the S(w) implementation against the
# derivation. It does NOT test the filters in section 2, which is where the risk is.
w_grid = np.arange(0, 120, 0.05)
observed_curve = np.array([survival(h, w) for w in w_grid])

area_under_curve = np.trapezoid(observed_curve, w_grid)
mean_wait_direct = (h ** 2).sum() / (2 * h.sum())
print(f'area under S(w) on [0, {w_grid.max():.0f}]  {area_under_curve:.4f}')
print(f'mean rider wait  Sh2 / 2Sh        {mean_wait_direct:.4f}')
print(f'difference                        {abs(area_under_curve - mean_wait_direct):.2e} min')
print(f'\n(residual is truncation of the integral at w = {w_grid.max():.0f};'
      f' max observed gap is {h.max():.0f} min)')

In [ ]:
even_curve = np.array([even_service_survival(mean_headway, w) for w in w_grid])
s10 = survival(h, PROMISED_HEADWAY_MIN)

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.fill_between(w_grid, even_curve, observed_curve, color=BLUE_L, alpha=0.45)
ax.plot(w_grid, observed_curve, lw=2.5, color=BLUE, label=f'route {ROUTE} as measured')
ax.plot(w_grid, even_curve, lw=2, color=AQUA,
        label=f'constant headway at the same mean')
ax.axvline(PROMISED_HEADWAY_MIN, color=ORANGE, lw=2, ls='--')
ax.plot([PROMISED_HEADWAY_MIN], [s10], 'o', ms=9, color=BLUE,
        markeredgecolor=SURFACE, markeredgewidth=2, zorder=5)
ax.annotate(f'S(10) = {s10:.1%}', xy=(PROMISED_HEADWAY_MIN, s10), xytext=(15.5, s10 + 0.14),
            color=INK, fontsize=10, arrowprops=dict(arrowstyle='-', color=MUTED, lw=1))

ax.set_xlim(0, 35); ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f'Route {ROUTE}: share of riders still waiting after w minutes')
ax.set_xlabel('w — minutes waited')
ax.legend(loc='upper right')
style(ax, 'share of riders waiting longer')
plt.show()

In [ ]:
headline = pd.Series({
    'mean headway (min)': mean_headway,
    'CV of headway': cv,
    'share of GAPS over 10 min': (h > PROMISED_HEADWAY_MIN).mean(),
    'share of RIDERS whose gap is over 10 min': h[h > PROMISED_HEADWAY_MIN].sum() / h.sum(),
    'S(10)  share of riders waiting over 10 min': s10,
    'S(15)': survival(h, 15),
    'S(20)': survival(h, 20),
    'mean rider wait (min)': mean_wait_direct,
})
print(headline.round(3).to_string())

The three "over 10 minutes" rows are three different quantities and are not
interchangeable:

- **share of gaps over 10 min** — one vote per gap, regardless of length.
- **share of riders whose gap exceeds 10 min** — each gap weighted by the riders it collects,
  which is proportional to its length.
- **`S(10)`** — the share who actually *wait* longer than 10 minutes. A rider landing in a
  20-minute gap waits past 10 only if they arrive in its first half.

Mean rider wait is included because it is the quantity the area-under-the-curve check tests,
not because it is a good summary — it compresses the whole curve to one number.

## 6. Breakdowns

In [ ]:
def promise_stats(frame):
    """S(10), mean headway and CV for one slice. NaN on slices under 500 gaps."""
    v = frame.headway_min.values
    v = v[v > 0]
    if len(v) < 500:
        return pd.Series({'S10': np.nan, 'mean_headway': np.nan, 'cv': np.nan, 'n': len(v)})
    return pd.Series({'S10': survival(v, PROMISED_HEADWAY_MIN),
                      'mean_headway': v.mean(),
                      'cv': v.std() / v.mean(),
                      'n': len(v)})


by_hour = promise_period.groupby('hour')[['headway_min']].apply(promise_stats)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))
axes[0].bar(by_hour.index, by_hour.S10, color=BLUE, width=0.85)
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title('S(10) by hour')
style(axes[0], 'riders waiting >10 min')

axes[1].bar(by_hour.index, by_hour.mean_headway, color=BLUE_L, width=0.85)
axes[1].axhline(PROMISED_HEADWAY_MIN, color=ORANGE, lw=2, ls='--')
axes[1].set_title('Mean headway by hour')
style(axes[1], 'minutes')

axes[2].bar(by_hour.index, by_hour.cv, color=AQUA, width=0.85)
axes[2].set_title('CV by hour')
style(axes[2], 'sd / mean')
for ax in axes:
    ax.set_xlabel('hour of day')
plt.tight_layout()
plt.show()

print(by_hour.round(3).to_string())

**To inspect:** whether the hour that minimises mean headway is the same hour that
minimises `S(10)`, and how `CV` moves across the day relative to both.

In [ ]:
# Along the route, one panel per direction component, ordered by position.
fig, axes = plt.subplots(len(components), 1, figsize=(11, 3.2 * len(components)))
axes = np.atleast_1d(axes)

for ax, (component_index, members) in zip(axes, enumerate(components)):
    anchor = max(members, key=lambda p: len(stops_by_pattern[p]))
    stop_order = (stop_visits[stop_visits.pid == anchor]
                  .groupby('stpid', observed=True).stop_sequence.median().sort_values())
    per_stop = (promise_period[promise_period.stpid.isin(stop_order.index)]
                .groupby('stpid', observed=True)[['headway_min']].apply(promise_stats)
                .reindex(stop_order.index).dropna())
    ax.plot(range(len(per_stop)), per_stop.S10, lw=2, color=BLUE, marker='o', ms=3)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f'S(10) by stop — direction component {component_index} '
                 f'(ordered along pattern {anchor}, {len(per_stop)} stops)')
    ax.set_xlabel('stops, in order along the pattern  →')
    style(ax, 'riders waiting >10 min')
plt.tight_layout()
plt.show()

**To inspect:** whether `S(10)` trends along the route, and whether the two directions
trend the same way. Note the x-axis is stop order along one anchor pattern, so stops served by
other patterns in the same component are placed by that anchor's sequence.

In [ ]:
# Weekends, held out of everything above. The promise covers them at 9a-9p.
weekend_period = headways[(headways.is_weekend) &
                          (headways.service_date >= PROMISE_BEGINS)]
h_weekend = weekend_period.headway_min.values
h_weekend = h_weekend[h_weekend > 0]

print(pd.DataFrame({
    'weekday': {'n': len(h), 'mean headway': h.mean(), 'CV': cv,
                'share gaps >10': (h > 10).mean(), 'S(10)': survival(h, 10)},
    'weekend': {'n': len(h_weekend), 'mean headway': h_weekend.mean(),
                'CV': h_weekend.std() / h_weekend.mean(),
                'share gaps >10': (h_weekend > 10).mean(),
                'S(10)': survival(h_weekend, 10)},
}).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
for values, label, colour in [(h, 'weekday', BLUE), (h_weekend, 'weekend', ORANGE)]:
    ax.plot(w_grid, [survival(values, w) for w in w_grid], lw=2.5, color=colour, label=label)
ax.axvline(PROMISED_HEADWAY_MIN, color=MUTED, lw=1.5, ls='--')
ax.set_xlim(0, 35)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Rider wait curve: weekday vs weekend')
ax.set_xlabel('w — minutes waited')
ax.legend()
style(ax, 'share waiting longer')
plt.show()

## 7. Unverified, untested, and known wrong

**Nothing in this notebook has been reviewed.** The list below is what to attack first.

### Errors already found and fixed here, as a calibration on the rest

- The direction check in §2 step 2 originally joined patterns sharing **any** stop. A few
  terminal-loop stops are used by both directions, so all 13 patterns merged into one
  component — and the check then compared that single group against nothing and printed a
  pass. **It could not fail.** It was then replaced by a shared-stop *count* threshold, which
  produced the right answer but left an arbitrary constant deciding direction. The current
  version correlates traversal order and carries an assertion that can fail.
- The destination sign was briefly treated as ground truth for direction. It is not: pattern
  `6982` is signed westbound and runs eastbound. Had the sign been adopted, one pattern would
  have been assigned the wrong direction and its stops would have been silently pooled with
  the opposing ones.
- An earlier reading of the raw files claimed ~1-minute polling. That was wrong: a day file
  holds many distinct poll minutes because routes are polled at different offsets. The
  per-vehicle figure in §3 is the one that matters.

### Where a mistake would hide without announcing itself

1. **Terminal exclusion (§2 step 4)** assumes `stop_sequence` minimum and maximum are the
   physical ends of the pattern. Not verified against stop locations.
2. **Both-direction stops are dropped, not split (§2 step 3).** Splitting each such stop by
   the direction of the arriving bus would keep the data. Not implemented.
3. **Window-then-difference ordering (§2 steps 5–6)** is what guarantees both ends of a gap
   sit inside the window. Worth confirming it does what the markdown claims.
4. **Day grouping** assumes no gap crosses midnight, which holds only because the window
   closes at 21:00.
5. **Effective sample size.** The millions of gaps are ~150 stops × ~400 days, and consecutive
   stops see the same buses. Nothing here reports an interval; any interval would have to be
   clustered, probably by day.
6. **`visits_per_vehicle` in §3 has mismatched scopes** — see the warning in that section.
   Needs rebuilding with the numerator and denominator on the same footing.
7. **Holidays are counted as ordinary weekdays.** Per `docs/fn-analysis-plan.md` §4b this
   needs a **calendar** holiday list, which does not exist.
   `data/derived/holiday_calendar.csv` is the *operational* list — a different object that
   misses weekend-dated holidays.
8. **Uniform arrivals** underpins every rider-weighted statistic. It is least defensible late
   in the evening.

### Not attempted

9. **No before/after comparison.** That is a difference statistic; the project rule requires a
   null on control routes first, plus the −4/+8 week washout window from
   `docs/fn-analysis-plan.md` §2.
10. **Scheduled vs realised.** `data/stopwatch/clean_timetables/` holds CTA's timetables in the
    same shape, so this identical computation runs on them. That separates "CTA didn't run the
    plan" from "the plan was never 10-minute service."
11. **The other 19 routes.** Scaling needs the corridor rule from the analysis plan
    (`X49`/`49B` → 49, `J14` → 14), since an express passing the same stop is a boardable bus.
    Route 66 has no sibling, so it does not arise here.
12. **Calibration near the 10-minute threshold.** `S(10)` depends on interpolation error at 10
    minutes specifically, not on average error. §3 compares eras; it does not size the residual
    on an individual gap.
13. **StopWatch's published validation ends July 2024**, before the entire promise period.